In [3]:
import re
import pandas as pd

# -------------------------------------------------
# 1️⃣ SQL 쿼리 파일 경로 (같은 폴더라면 파일명만)
# -------------------------------------------------
sql_file = "insert_data.sql"  # 예: "insert_data.sql"

# -------------------------------------------------
# 2️⃣ INSERT문 파싱용 정규식
# -------------------------------------------------
pattern = re.compile(
    r"INSERT\s+(?:IGNORE\s+)?INTO\s+`(?P<table>\w+)`\s*\((?P<columns>[^)]+)\)\s*VALUES\s*(?P<values>[^;]+);?",
    re.IGNORECASE | re.DOTALL,
)

# -------------------------------------------------
# 3️⃣ SQL 파일 읽기
# -------------------------------------------------
with open(sql_file, "r", encoding="utf-8") as f:
    sql_text = f.read()

tables = {}

# -------------------------------------------------
# 4️⃣ 각 INSERT문에서 테이블명·컬럼·값 추출
# -------------------------------------------------
for match in pattern.finditer(sql_text):
    table_name = match.group("table")
    columns = [col.strip(" `") for col in match.group("columns").split(",")]
    values_block = match.group("values")

    # 각 행별 데이터 추출
    rows = re.findall(r"\(([^)]+)\)", values_block)
    data = []
    for row in rows:
        # 쉼표 기준 분리 + 따옴표/NULL 처리
        parts = [x.strip().strip("'") for x in row.split(",")]
        parts = [None if x.upper() == "NULL" else x for x in parts]
        data.append(parts)

    df = pd.DataFrame(data, columns=columns)
    tables[table_name] = df

# -------------------------------------------------
# 5️⃣ 데이터 미리보기
# -------------------------------------------------
for name, df in tables.items():
    print(f"\n📘 테이블명: {name}")
    display(df.head())

# -------------------------------------------------
# 6️⃣ CSV로 저장 (테이블별 파일)
# -------------------------------------------------
for name, df in tables.items():
    filename = f"{name}.csv"
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"✅ 저장 완료: {filename}")


📘 테이블명: gnr_road_info


,DTCT_UID,DTCT_STT,DTCT_WORK,DTCT_PD,DTCT_FN_PD,DTCT_FN_CONTENTS
0,gnsoft1_250714084125_11277,2,None,2025-07-14 08:41:25,None,None
1,gnsoft1_251106093349_24,0,None,2025-11-06 09:33:49,None,None
2,gnsoft1_251106093431_32,0,None,2025-11-06 09:34:31,None,None
3,gnsoft1_251106194824_90,0,None,2025-11-06 19:48:24,None,None
4,gnsoft2_250712155250_39377,2,None,2025-07-12 15:52:51,None,None



📘 테이블명: gnr_road_info_mtdt


,MTDT_INDEX,DTCT_UID,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO,IMG_CONF
0,2603,RAD01_240119162357_441,1,0,36.384627,127.3434593,대전광역시 유성구 신성동 산 26-12,신성동,신성동,None
1,2613,RAD01_240119162610_0,1,0,36.398713,127.3578639,대전광역시 유성구 장동 202-11,장동,신성동,None
2,2614,RAD01_240119162611_0,1,0,36.3987826,127.3580054,대전광역시 유성구 장동 202-11,장동,신성동,None
3,2655,RAD01_240119163618_0,1,0,36.4278992,127.387323,대전광역시 유성구 관평동 1286,관평동,관평동,None
4,2659,RAD01_240119163629_608,1,0,36.4277407,127.3877235,대전광역시 유성구 관평동 1286,관평동,관평동,None


✅ 저장 완료: gnr_road_info.csv
✅ 저장 완료: gnr_road_info_mtdt.csv


In [4]:
import pandas as pd

# -------------------------------------------------
# 1️⃣ 파일 불러오기
# -------------------------------------------------
road_info = pd.read_csv("gnr_road_info.csv", encoding="utf-8-sig")
road_mtdt = pd.read_csv("gnr_road_info_mtdt.csv", encoding="utf-8-sig")

# -------------------------------------------------
# 2️⃣ 데이터 기본 구조 확인
# -------------------------------------------------
print("=== gnr_road_info.csv ===")
display(road_info.head())
print("\n컬럼 목록:", road_info.columns.tolist())
print("행 개수:", len(road_info))
print("결측치 개수:\n", road_info.isnull().sum())

print("\n=== gnr_road_info_mtdt.csv ===")
display(road_mtdt.head())
print("\n컬럼 목록:", road_mtdt.columns.tolist())
print("행 개수:", len(road_mtdt))
print("결측치 개수:\n", road_mtdt.isnull().sum())

# -------------------------------------------------
# 3️⃣ 공통 키(DTCT_UID) 중복 및 일치도 분석
# -------------------------------------------------
if "DTCT_UID" in road_info.columns and "DTCT_UID" in road_mtdt.columns:
    print("\n🔍 공통 키(DTCT_UID) 중복 여부")
    print("road_info 중복 수:", road_info["DTCT_UID"].duplicated().sum())
    print("road_mtdt 중복 수:", road_mtdt["DTCT_UID"].duplicated().sum())

    print("\n🔗 road_info의 UID가 road_mtdt에도 존재하는 비율:")
    matched = road_info["DTCT_UID"].isin(road_mtdt["DTCT_UID"]).mean() * 100
    print(f"   → {matched:.2f}%")

# -------------------------------------------------
# 4️⃣ 수치형 컬럼 기본 통계
# -------------------------------------------------
print("\n📊 기본 통계 (수치형 컬럼)")
display(road_info.describe(include="number"))
display(road_mtdt.describe(include="number"))

=== gnr_road_info.csv ===


,DTCT_UID,DTCT_STT,DTCT_WORK,DTCT_PD,DTCT_FN_PD,DTCT_FN_CONTENTS
0,gnsoft1_250714084125_11277,2,NaN,2025-07-14 08:41:25,NaN,NaN
1,gnsoft1_251106093349_24,0,NaN,2025-11-06 09:33:49,NaN,NaN
2,gnsoft1_251106093431_32,0,NaN,2025-11-06 09:34:31,NaN,NaN
3,gnsoft1_251106194824_90,0,NaN,2025-11-06 19:48:24,NaN,NaN
4,gnsoft2_250712155250_39377,2,NaN,2025-07-12 15:52:51,NaN,NaN



컬럼 목록: ['DTCT_UID', 'DTCT_STT', 'DTCT_WORK', 'DTCT_PD', 'DTCT_FN_PD', 'DTCT_FN_CONTENTS']
행 개수: 1079
결측치 개수:
 DTCT_UID               0
DTCT_STT               0
DTCT_WORK           1072
DTCT_PD                0
DTCT_FN_PD          1071
DTCT_FN_CONTENTS    1072
dtype: int64

=== gnr_road_info_mtdt.csv ===


,MTDT_INDEX,DTCT_UID,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO,IMG_CONF
0,2603,RAD01_240119162357_441,1,0,36.384627,127.343459,대전광역시 유성구 신성동 산 26-12,신성동,신성동,NaN
1,2613,RAD01_240119162610_0,1,0,36.398713,127.357864,대전광역시 유성구 장동 202-11,장동,신성동,NaN
2,2614,RAD01_240119162611_0,1,0,36.398783,127.358005,대전광역시 유성구 장동 202-11,장동,신성동,NaN
3,2655,RAD01_240119163618_0,1,0,36.427899,127.387323,대전광역시 유성구 관평동 1286,관평동,관평동,NaN
4,2659,RAD01_240119163629_608,1,0,36.427741,127.387724,대전광역시 유성구 관평동 1286,관평동,관평동,NaN



컬럼 목록: ['MTDT_INDEX', 'DTCT_UID', 'ROAD_POTHOLE', 'ROAD_CRACK', 'PLC_LTTD', 'PLC_LGTD', 'ROAD_INFO', 'DSTA_LGLD_INFO', 'DSTA_ADSTRD_INFO', 'IMG_CONF']
행 개수: 1079
결측치 개수:
 MTDT_INDEX            0
DTCT_UID              0
ROAD_POTHOLE          0
ROAD_CRACK            0
PLC_LTTD              0
PLC_LGTD              0
ROAD_INFO             0
DSTA_LGLD_INFO        0
DSTA_ADSTRD_INFO      0
IMG_CONF            579
dtype: int64

🔍 공통 키(DTCT_UID) 중복 여부
road_info 중복 수: 0
road_mtdt 중복 수: 0

🔗 road_info의 UID가 road_mtdt에도 존재하는 비율:
   → 100.00%

📊 기본 통계 (수치형 컬럼)


,DTCT_STT
count,1079.000000
mean,0.617238
std,0.806352
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,3.000000


,MTDT_INDEX,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,IMG_CONF
count,1079.000000,1079.0,1079.0,1079.000000,1079.000000,500.000000
mean,12773.863763,1.0,0.0,36.369390,127.358201,0.815176
std,4580.703636,0.0,0.0,0.031670,0.041875,0.064584
min,2603.000000,1.0,0.0,36.266938,127.256012,0.601800
25%,9211.000000,1.0,0.0,36.352941,127.323617,0.808050
50%,13446.000000,1.0,0.0,36.366768,127.344982,0.838400
75%,16705.000000,1.0,0.0,36.390128,127.402395,0.851400
max,18814.000000,1.0,0.0,36.465281,127.462940,0.909200


In [5]:
import pandas as pd

# 파일 불러오기
road_info = pd.read_csv("gnr_road_info.csv", encoding="utf-8-sig")
road_mtdt = pd.read_csv("gnr_road_info_mtdt.csv", encoding="utf-8-sig")

# -------------------------------------------------
# 1️⃣ DTCT_UID 기준 병합
# -------------------------------------------------
merged = pd.merge(road_info, road_mtdt, on="DTCT_UID", how="inner")

# -------------------------------------------------
# 2️⃣ 병합 결과 확인
# -------------------------------------------------
print("✅ 병합된 데이터 개수:", len(merged))
print("컬럼 수:", len(merged.columns))
display(merged.head())

# -------------------------------------------------
# 3️⃣ CSV로 저장
# -------------------------------------------------
merged.to_csv("merged_road_data.csv", index=False, encoding="utf-8-sig")
print("💾 병합 결과 저장 완료: merged_road_data.csv")

✅ 병합된 데이터 개수: 1079
컬럼 수: 15


,DTCT_UID,DTCT_STT,DTCT_WORK,DTCT_PD,DTCT_FN_PD,DTCT_FN_CONTENTS,MTDT_INDEX,ROAD_POTHOLE,ROAD_CRACK,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO,IMG_CONF
0,gnsoft1_250714084125_11277,2,NaN,2025-07-14 08:41:25,NaN,NaN,17249,1,0,36.346661,127.353838,대전광역시 서구 월평동 114-4,월평동,월평1동,0.8023
1,gnsoft1_251106093349_24,0,NaN,2025-11-06 09:33:49,NaN,NaN,18489,1,0,36.390977,127.407906,대전광역시 유성구 문지동 661-3,문지동,전민동,0.8034
2,gnsoft1_251106093431_32,0,NaN,2025-11-06 09:34:31,NaN,NaN,18490,1,0,36.390877,127.407925,대전광역시 유성구 문지동 661-3,문지동,전민동,0.8120
3,gnsoft1_251106194824_90,0,NaN,2025-11-06 19:48:24,NaN,NaN,18509,1,0,36.350716,127.368103,대전광역시 서구 갈마동 690,갈마동,갈마1동,0.8014
4,gnsoft2_250712155250_39377,2,NaN,2025-07-12 15:52:51,NaN,NaN,17265,1,0,36.348692,127.335768,대전광역시 유성구 상대동 556,상대동,온천1동,0.8374


💾 병합 결과 저장 완료: merged_road_data.csv


In [7]:
# 제거할 컬럼 목록
drop_cols = [
    "DTCT_STT",
    "DTCT_WORK",
    "DTCT_FN_PD",
    "DTCT_FN_CONTENTS",
    "MTDT_INDEX",
    "ROAD_POTHOLE",
    "ROAD_CRACK",
    "IMG_CONF",
]

# -------------------------------------------------
# 1️⃣ 해당 컬럼 삭제 (존재하지 않으면 무시)
# -------------------------------------------------
cleaned = merged.drop(columns=drop_cols, errors="ignore")

# -------------------------------------------------
# 2️⃣ 결과 확인
# -------------------------------------------------
print("✅ 컬럼 삭제 후 데이터프레임 크기:", cleaned.shape)
display(cleaned.head())

# -------------------------------------------------
# 3️⃣ CSV 저장
# -------------------------------------------------
cleaned.to_csv("road_data.csv", index=False, encoding="utf-8-sig")
print("💾 저장 완료: road_data.csv")

✅ 컬럼 삭제 후 데이터프레임 크기: (1079, 7)


,DTCT_UID,DTCT_PD,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO
0,gnsoft1_250714084125_11277,2025-07-14 08:41:25,36.346661,127.353838,대전광역시 서구 월평동 114-4,월평동,월평1동
1,gnsoft1_251106093349_24,2025-11-06 09:33:49,36.390977,127.407906,대전광역시 유성구 문지동 661-3,문지동,전민동
2,gnsoft1_251106093431_32,2025-11-06 09:34:31,36.390877,127.407925,대전광역시 유성구 문지동 661-3,문지동,전민동
3,gnsoft1_251106194824_90,2025-11-06 19:48:24,36.350716,127.368103,대전광역시 서구 갈마동 690,갈마동,갈마1동
4,gnsoft2_250712155250_39377,2025-07-12 15:52:51,36.348692,127.335768,대전광역시 유성구 상대동 556,상대동,온천1동


💾 저장 완료: road_data.csv


In [8]:
import pandas as pd

df = pd.read_csv("road_data.csv", encoding="utf-8-sig")

print("행 개수:", len(df))
print("컬럼 목록:", df.columns.tolist())

display(df.head())

# 기본 정보 요약
df.info()

# 결측치 요약
print("\n결측치 개수:")
print(df.isnull().sum())

# 수치형 통계
display(df.describe(include="number"))

행 개수: 1079
컬럼 목록: ['DTCT_UID', 'DTCT_PD', 'PLC_LTTD', 'PLC_LGTD', 'ROAD_INFO', 'DSTA_LGLD_INFO', 'DSTA_ADSTRD_INFO']


,DTCT_UID,DTCT_PD,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO
0,gnsoft1_250714084125_11277,2025-07-14 08:41:25,36.346661,127.353838,대전광역시 서구 월평동 114-4,월평동,월평1동
1,gnsoft1_251106093349_24,2025-11-06 09:33:49,36.390977,127.407906,대전광역시 유성구 문지동 661-3,문지동,전민동
2,gnsoft1_251106093431_32,2025-11-06 09:34:31,36.390877,127.407925,대전광역시 유성구 문지동 661-3,문지동,전민동
3,gnsoft1_251106194824_90,2025-11-06 19:48:24,36.350716,127.368103,대전광역시 서구 갈마동 690,갈마동,갈마1동
4,gnsoft2_250712155250_39377,2025-07-12 15:52:51,36.348692,127.335768,대전광역시 유성구 상대동 556,상대동,온천1동


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1079 entries, 0 to 1078
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   DTCT_UID          1079 non-null   object 
 1   DTCT_PD           1079 non-null   object 
 2   PLC_LTTD          1079 non-null   float64
 3   PLC_LGTD          1079 non-null   float64
 4   ROAD_INFO         1079 non-null   object 
 5   DSTA_LGLD_INFO    1079 non-null   object 
 6   DSTA_ADSTRD_INFO  1079 non-null   object 
dtypes: float64(2), object(5)
memory usage: 59.1+ KB

결측치 개수:
DTCT_UID            0
DTCT_PD             0
PLC_LTTD            0
PLC_LGTD            0
ROAD_INFO           0
DSTA_LGLD_INFO      0
DSTA_ADSTRD_INFO    0
dtype: int64


,PLC_LTTD,PLC_LGTD
count,1079.000000,1079.000000
mean,36.369390,127.358201
std,0.031670,0.041875
min,36.266938,127.256012
25%,36.352941,127.323617
50%,36.366768,127.344982
75%,36.390128,127.402395
max,36.465281,127.462940


In [9]:
import pandas as pd

# -------------------------------------------------
# 1️⃣ 파일 불러오기
# -------------------------------------------------
df = pd.read_csv("road_data.csv", encoding="utf-8-sig")

# -------------------------------------------------
# 2️⃣ 날짜 컬럼을 datetime으로 변환
# -------------------------------------------------
df["DTCT_PD"] = pd.to_datetime(df["DTCT_PD"], errors="coerce")

# -------------------------------------------------
# 3️⃣ 날짜 내림차순 정렬
# -------------------------------------------------
df = df.sort_values(by="DTCT_PD", ascending=False).reset_index(drop=True)

# -------------------------------------------------
# 4️⃣ 정렬 결과 확인
# -------------------------------------------------
display(df.head())

# -------------------------------------------------
# 5️⃣ 같은 파일명으로 덮어쓰기 저장
# -------------------------------------------------
df.to_csv("road_data.csv", index=False, encoding="utf-8-sig")
print("✅ road_data.csv 파일이 날짜 기준 내림차순으로 정렬되어 저장되었습니다.")

,DTCT_UID,DTCT_PD,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO
0,gnsoft2_251111083550_192308,2025-11-11 08:35:51,36.377844,127.410806,대전광역시 대덕구 읍내동 505-3,읍내동,회덕동
1,gnsoft2_251111082751_188445,2025-11-11 08:27:51,36.368095,127.413937,대전광역시 대덕구 대화동 38-39,대화동,대화동
2,gnsoft2_251111082737_188303,2025-11-11 08:27:38,36.367703,127.415327,대전광역시 대덕구 대화동 37-9,대화동,대화동
3,gnsoft2_251111082724_188111,2025-11-11 08:27:25,36.367781,127.417822,대전광역시 대덕구 대화동 16-333,대화동,대화동
4,gnsoft2_251111082339_186325,2025-11-11 08:23:39,36.359762,127.417344,대전광역시 대덕구 오정동 40-9,오정동,오정동


✅ road_data.csv 파일이 날짜 기준 내림차순으로 정렬되어 저장되었습니다.


In [10]:
import pandas as pd

# -------------------------------------------------
# 1️⃣ 파일 불러오기
# -------------------------------------------------
df = pd.read_csv("road_data.csv", encoding="utf-8-sig")

# -------------------------------------------------
# 2️⃣ 날짜형 변환
# -------------------------------------------------
df["DTCT_PD"] = pd.to_datetime(df["DTCT_PD"], errors="coerce")

# -------------------------------------------------
# 3️⃣ 2025년 11월 6일 이후 데이터 제거
# -------------------------------------------------
cutoff_date = pd.Timestamp("2025-11-06")
filtered_df = df[df["DTCT_PD"] < cutoff_date].reset_index(drop=True)

# -------------------------------------------------
# 4️⃣ 결과 확인
# -------------------------------------------------
print(f"🧾 원본 행 수: {len(df)}")
print(f"🧹 삭제 후 행 수: {len(filtered_df)}")

# -------------------------------------------------
# 5️⃣ 덮어쓰기 저장
# -------------------------------------------------
filtered_df.to_csv("road_data.csv", index=False, encoding="utf-8-sig")
print("✅ 2025-11-06 이후 데이터가 삭제된 road_data.csv 파일로 저장 완료되었습니다.")

🧾 원본 행 수: 1079
🧹 삭제 후 행 수: 972
✅ 2025-11-06 이후 데이터가 삭제된 road_data.csv 파일로 저장 완료되었습니다.


In [1]:
# =====================================================================
# 🧭 road_data.csv EDA (Plotly) — 월/시간대 전체 표시 버전
# =====================================================================

import pandas as pd
import plotly.express as px
import folium

# -------------------------------------------------
# 1️⃣ 데이터 불러오기 및 전처리
# -------------------------------------------------
df = pd.read_csv("road_data.csv", encoding="utf-8-sig")
df["DTCT_PD"] = pd.to_datetime(df["DTCT_PD"], errors="coerce")

# 2025년 7월 데이터 제거
df = df[~((df["DTCT_PD"].dt.year == 2025) & (df["DTCT_PD"].dt.month == 7))].reset_index(
    drop=True
)

# -------------------------------------------------
# 2️⃣ 파생 컬럼 추가
# -------------------------------------------------
df["연도"] = df["DTCT_PD"].dt.year
df["월"] = df["DTCT_PD"].dt.month
df["시간대"] = df["DTCT_PD"].dt.hour

day_map = {
    "Monday": "월요일",
    "Tuesday": "화요일",
    "Wednesday": "수요일",
    "Thursday": "목요일",
    "Friday": "금요일",
    "Saturday": "토요일",
    "Sunday": "일요일",
}
df["요일"] = df["DTCT_PD"].dt.day_name().map(day_map)

# -------------------------------------------------
# 3️⃣ 월별 탐지 추세 (모든 월 표시)
# -------------------------------------------------
monthly = (
    df["월"]
    .value_counts()
    .sort_index()
    .reindex(range(1, 13), fill_value=0)
    .reset_index()
)
monthly.columns = ["월", "건수"]

fig1 = px.bar(
    monthly,
    x="월",
    y="건수",
    title="📅 월별 탐지 건수",
    color="건수",
    color_continuous_scale="Blues",
    text="건수",
)
fig1.update_traces(textposition="outside")
fig1.update_layout(xaxis=dict(dtick=1), xaxis_title="월", yaxis_title="건수")
fig1.show()

# -------------------------------------------------
# 4️⃣ 요일별 탐지 추세
# -------------------------------------------------
order = ["월요일", "화요일", "수요일", "목요일", "금요일", "토요일", "일요일"]
weekday = df["요일"].value_counts().reindex(order).fillna(0).reset_index()
weekday.columns = ["요일", "건수"]

fig2 = px.bar(
    weekday,
    x="요일",
    y="건수",
    title="📆 요일별 탐지 건수",
    color="건수",
    color_continuous_scale="Sunset",
    text="건수",
)
fig2.update_traces(textposition="outside")
fig2.update_layout(xaxis_title="요일", yaxis_title="건수")
fig2.show()

# -------------------------------------------------
# 5️⃣ 시간대별 탐지 추세 (0~23 전체 표시)
# -------------------------------------------------
hourly = (
    df["시간대"]
    .value_counts()
    .sort_index()
    .reindex(range(0, 24), fill_value=0)
    .reset_index()
)
hourly.columns = ["시간대", "건수"]

fig3 = px.bar(
    hourly,
    x="시간대",
    y="건수",
    title="⏰ 시간대별 탐지 건수",
    color="건수",
    color_continuous_scale="Purples",
    text="건수",
)
fig3.update_traces(textposition="outside")
fig3.update_layout(xaxis=dict(dtick=1), xaxis_title="시간대 (시)", yaxis_title="건수")
fig3.show()

# -------------------------------------------------
# 6️⃣ 행정동별 탐지 건수
# -------------------------------------------------
area = df["DSTA_ADSTRD_INFO"].value_counts().reset_index()
area.columns = ["행정동", "건수"]

fig4 = px.bar(
    area,
    y="행정동",
    x="건수",
    orientation="h",
    title="🏙️ 행정동별 탐지 건수",
    color="건수",
    color_continuous_scale="Viridis",
    text="건수",
)
fig4.update_traces(textposition="outside")
fig4.update_layout(xaxis_title="건수", yaxis_title="행정동")
fig4.show()

# -------------------------------------------------
# 7️⃣ 시간대 × 행정동 히트맵
# -------------------------------------------------
pivot = df.pivot_table(
    index="시간대",
    columns="DSTA_ADSTRD_INFO",
    values="DTCT_UID",
    aggfunc="count",
    fill_value=0,
)
pivot = pivot.reindex(index=range(0, 24), fill_value=0)  # 모든 시간대 포함

fig5 = px.imshow(
    pivot.T,
    color_continuous_scale="YlOrRd",
    title="🔥 시간대별 · 행정동별 탐지 패턴 (Heatmap)",
    labels=dict(x="시간대", y="행정동", color="탐지 건수"),
)
fig5.update_layout(height=600, width=900)
fig5.show()

# -------------------------------------------------
# 8️⃣ 지도 시각화 (Folium)
# -------------------------------------------------
center_lat, center_lon = df["PLC_LTTD"].mean(), df["PLC_LGTD"].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["PLC_LTTD"], row["PLC_LGTD"]],
        radius=3,
        color="red",
        fill=True,
        fill_opacity=0.6,
        popup=row["ROAD_INFO"],
    ).add_to(m)

print("🗺️ 지도 시각화 준비 완료 — 아래 지도 확인")
m

🗺️ 지도 시각화 준비 완료 — 아래 지도 확인


In [2]:
import pandas as pd
import numpy as np
from math import floor

# 탐지 데이터 로드
df = pd.read_csv("road_data.csv", encoding="utf-8-sig")

# 위경도를 일정 단위(예: 0.001도 ≈ 약 100m)로 구간화
grid_size = 0.001

df["lat_bin"] = (df["PLC_LTTD"] / grid_size).apply(floor)
df["lon_bin"] = (df["PLC_LGTD"] / grid_size).apply(floor)
df["segment_id"] = df["lat_bin"].astype(str) + "_" + df["lon_bin"].astype(str)

# 구간별 대표 위치 및 파손 여부 집계
segment = (
    df.groupby("segment_id")
    .agg(
        중심위도=("PLC_LTTD", "mean"),
        중심경도=("PLC_LGTD", "mean"),
        탐지건수=("DTCT_UID", "count"),
    )
    .reset_index()
)

# 구간별 파손 라벨(탐지 1건 이상이면 파손=1)
segment["파손발생"] = np.where(segment["탐지건수"] > 0, 1, 0)

# 불필요 컬럼 제거
segment = segment[["segment_id", "중심위도", "중심경도", "파손발생"]]

segment.to_csv("road_segment_label.csv", index=False, encoding="utf-8-sig")
print("✅ 구간 단위 파손 라벨 테이블 생성 완료: road_segment_label.csv")
display(segment.head())

✅ 구간 단위 파손 라벨 테이블 생성 완료: road_segment_label.csv


,segment_id,중심위도,중심경도,파손발생
0,36266_127284,36.266938,127.284043,1
1,36279_127256,36.279881,127.256012,1
2,36285_127260,36.285896,127.260768,1
3,36285_127307,36.285658,127.307760,1
4,36285_127310,36.285338,127.310041,1


In [6]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString
from tqdm import tqdm

tqdm.pandas()

# -------------------------------------------------
# 1️⃣ 파일 경로 및 SHP 로드
# -------------------------------------------------
shp_path = "/home/user/ai/AnKiHyeon/어프렌티스/국토교통부_도로(현황)/C_UQ151.shp"
damage_path = "road_segment_label.csv"

# 인코딩 자동 감지
for enc in ["utf-8", "euc-kr", "cp949"]:
    try:
        roads = gpd.read_file(shp_path, encoding=enc)
        print(f"✅ 인코딩 성공: {enc}")
        break
    except UnicodeDecodeError:
        continue

# CRS 변환
roads = roads.to_crs(epsg=4326)

# 컬럼 리네임
roads = roads.rename(
    columns={
        "DGM_NM": "ROAD_NAME",
        "GRAD_SE": "ROAD_GRADE",
        "ROAD_NO": "ROAD_NO",
        "SIGNGU_SE": "ADM_CD",
    }
)
roads = roads[["ROAD_NAME", "ROAD_GRADE", "ROAD_NO", "ADM_CD", "geometry"]]

# -------------------------------------------------
# 2️⃣ POLYGON → 중심선 (centroid 기반)
# -------------------------------------------------
roads["geometry"] = roads["geometry"].centroid
roads = roads.dropna(subset=["geometry"])
roads = roads.reset_index(drop=True)
roads["segment_id"] = "SHP_" + roads.index.astype(str)
print(f"📍 총 도로구역: {len(roads)}개 (폴리곤 → 중심선 변환 완료)")

# -------------------------------------------------
# 3️⃣ 탐지 데이터 로드 (파손 구간)
# -------------------------------------------------
damage_df = pd.read_csv(damage_path, encoding="utf-8-sig")
damage_df["geometry"] = gpd.points_from_xy(damage_df["중심경도"], damage_df["중심위도"])
damage_gdf = gpd.GeoDataFrame(damage_df, geometry="geometry", crs="EPSG:4326")

# -------------------------------------------------
# 4️⃣ 공간 조인 (intersects)
# -------------------------------------------------
joined = gpd.sjoin(roads, damage_gdf, how="left", predicate="intersects")
joined["파손발생"] = np.where(joined["DTCT_UID"].notna(), 1, 0)

# -------------------------------------------------
# 5️⃣ 비파손 구간 랜덤 샘플링
# -------------------------------------------------
damage_count = sum(joined["파손발생"] == 1)
non_damage_df = joined[joined["파손발생"] == 0].sample(n=damage_count, random_state=42)
damage_df = joined[joined["파손발생"] == 1]

balanced = pd.concat([damage_df, non_damage_df], ignore_index=True)
balanced = balanced[
    [
        "segment_id",
        "ROAD_NAME",
        "ROAD_GRADE",
        "ROAD_NO",
        "ADM_CD",
        "파손발생",
        "geometry",
    ]
]

# -------------------------------------------------
# 6️⃣ 결과 저장
# -------------------------------------------------
balanced_gdf = gpd.GeoDataFrame(balanced, geometry="geometry", crs="EPSG:4326")
balanced_gdf.to_file("road_segment_balanced.shp", encoding="utf-8")
balanced_gdf.drop(columns="geometry").to_csv(
    "road_segment_balanced.csv", index=False, encoding="utf-8-sig"
)

print("✅ road_segment_balanced.csv / .shp 생성 완료")
print(f"   파손(1): {len(damage_df)}, 비파손(0): {len(non_damage_df)}")

/home/user/venvs/AnKiHyeon_tensorflow/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: /home/user/ai/AnKiHyeon/어프렌티스/국토교통부_도로(현황)/C_UQ151.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


✅ 인코딩 성공: euc-kr


/tmp/ipykernel_429837/921181828.py:41: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  roads["geometry"] = roads["geometry"].centroid


📍 총 도로구역: 260743개 (폴리곤 → 중심선 변환 완료)


KeyError: 'DTCT_UID'